# RecA Feature Selection and RFE Benchmark Workflow

This notebook combines the uploaded feature-selection files into one reproducible workflow:

1. Low-variance filtering and ANOVA F-score feature ranking.
2. Existing feature-selection notebook notes/code.
3. SVM-RFE, RF-RFE, and XGBoost-RFE benchmarking.

Run this notebook after generating `outputs/02_recA_modeling_matrix.csv` from the previous RecA PaDEL fingerprint workflow.

## Input Files Combined

- `03_Feature_selection_low_variance_Fscore.py`
- `03_Features_selection.ipynb`
- `03a_RFE_feature_selection_benchmark.py`

## Stage 03 — Low-Variance Filtering and ANOVA F-score

The following cells contain the full converted Python workflow for low-variance filtering and F-score ranking.

In [7]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif


# SCRIPT_DIR = Path(__file__).resolve().parent
SCRIPT_DIR = Path.cwd()

INPUT_FILE = SCRIPT_DIR / "outputs" / "02_recA_modeling_matrix.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs" / "feature_selection"

LOW_VARIANCE_FILE = OUTPUT_DIR / "03_recA_low_variance_filtered.csv"
FSCORE_FILE = OUTPUT_DIR / "03_recA_fscore_ranking.csv"
TOP_FEATURE_FILE = OUTPUT_DIR / "03_recA_top_200_fscore.csv"

META_COLUMNS = ["Name", "canonical_smiles", "bioactivity_class", "class"]

DEFAULT_VARIANCE_THRESHOLD = 0.16
DEFAULT_TOP_K = 200


def load_dataset() -> pd.DataFrame:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Please run 02 fingerprint matrix assembly first."
        )

    df = pd.read_csv(INPUT_FILE)

    missing = [col for col in META_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing expected metadata columns: {missing}")

    return df


def prepare_feature_matrix(
    df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:

    meta = df[META_COLUMNS].copy()

    feature_columns = [
        col for col in df.columns
        if col not in META_COLUMNS
    ]

    x = (
        df[feature_columns]
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    y = meta["class"].astype(int)

    return meta, x, y


def remove_low_variance_features(
    x: pd.DataFrame,
    threshold: float,
) -> tuple[pd.DataFrame, VarianceThreshold]:

    selector = VarianceThreshold(threshold=threshold)
    transformed = selector.fit_transform(x)

    selected_columns = x.columns[
        selector.get_support()
    ].tolist()

    x_reduced = pd.DataFrame(
        transformed,
        columns=selected_columns,
        index=x.index,
    )

    return x_reduced, selector


def score_features_by_fscore(
    x: pd.DataFrame,
    y: pd.Series,
) -> pd.DataFrame:

    selector = SelectKBest(
        score_func=f_classif,
        k="all",
    )

    selector.fit(x, y)

    scores = pd.DataFrame(
        {
            "feature": x.columns,
            "f_score": selector.scores_,
            "p_value": selector.pvalues_,
        }
    )

    scores = scores.replace([np.inf, -np.inf], np.nan)
    scores["f_score"] = scores["f_score"].fillna(-1.0)
    scores["p_value"] = scores["p_value"].fillna(1.0)

    scores = (
        scores
        .sort_values(
            ["f_score", "p_value"],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

    return scores


def build_top_feature_dataset(
    meta: pd.DataFrame,
    x_reduced: pd.DataFrame,
    feature_scores: pd.DataFrame,
    top_k: int,
) -> tuple[pd.DataFrame, list[str]]:

    top_k = min(top_k, len(feature_scores))

    selected_features = (
        feature_scores
        .head(top_k)["feature"]
        .tolist()
    )

    top_dataset = pd.concat(
        [
            meta,
            x_reduced[selected_features],
        ],
        axis=1,
    )

    return top_dataset, selected_features


def save_outputs(
    meta: pd.DataFrame,
    x_reduced: pd.DataFrame,
    feature_scores: pd.DataFrame,
    top_dataset: pd.DataFrame,
) -> None:

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    low_variance_dataset = pd.concat(
        [meta, x_reduced],
        axis=1,
    )

    low_variance_dataset.to_csv(
        LOW_VARIANCE_FILE,
        index=False,
    )

    feature_scores.to_csv(
        FSCORE_FILE,
        index=False,
    )

    top_dataset.to_csv(
        TOP_FEATURE_FILE,
        index=False,
    )


def main() -> None:
    parser = argparse.ArgumentParser(
        description=(
            "Remove low-variance fingerprint features and rank "
            "the remaining descriptors using ANOVA F-score."
        )
    )

    parser.add_argument(
        "--variance-threshold",
        type=float,
        default=DEFAULT_VARIANCE_THRESHOLD,
        help="VarianceThreshold cutoff for binary fingerprints.",
    )

    parser.add_argument(
        "--top-k",
        type=int,
        default=DEFAULT_TOP_K,
        help="Number of top F-score-ranked features to keep.",
    )

    args = parser.parse_args()

    df = load_dataset()

    meta, x, y = prepare_feature_matrix(df)

    x_reduced, _ = remove_low_variance_features(
        x,
        threshold=args.variance_threshold,
    )

    feature_scores = score_features_by_fscore(
        x_reduced,
        y,
    )

    top_dataset, selected_features = build_top_feature_dataset(
        meta,
        x_reduced,
        feature_scores,
        top_k=args.top_k,
    )

    save_outputs(
        meta,
        x_reduced,
        feature_scores,
        top_dataset,
    )

    print("\nFeature selection completed successfully.")
    print(f"Input shape: {df.shape}")
    print(f"Original feature count: {x.shape[1]}")
    print(f"After low-variance filtering: {x_reduced.shape[1]}")
    print(f"Selected top-k features: {len(selected_features)}")

    print("\nClass distribution:")
    print(meta["bioactivity_class"].value_counts().to_string())

    print("\nTop 10 F-score features:")
    print(feature_scores.head(10).to_string(index=False))

    print(f"\nLow-variance output:\n{LOW_VARIANCE_FILE}")
    print(f"\nF-score ranking output:\n{FSCORE_FILE}")
    print(f"\nTop feature dataset output:\n{TOP_FEATURE_FILE}")

In [8]:

# Notebook runner for Stage 03: Low-variance filtering + ANOVA F-score
variance_threshold = DEFAULT_VARIANCE_THRESHOLD
top_k = DEFAULT_TOP_K

df = load_dataset()
meta, x, y = prepare_feature_matrix(df)

x_reduced, variance_selector = remove_low_variance_features(
    x,
    threshold=variance_threshold,
)

feature_scores = score_features_by_fscore(x_reduced, y)

top_dataset, selected_features = build_top_feature_dataset(
    meta,
    x_reduced,
    feature_scores,
    top_k=top_k,
)

save_outputs(
    meta,
    x_reduced,
    feature_scores,
    top_dataset,
)

print("\nFeature selection completed successfully.")
print(f"Input shape: {df.shape}")
print(f"Original feature count: {x.shape[1]}")
print(f"After low-variance filtering: {x_reduced.shape[1]}")
print(f"Selected top-k features: {len(selected_features)}")

print("\nClass distribution:")
print(meta["bioactivity_class"].value_counts().to_string())

print("\nTop 10 F-score features:")
display(feature_scores.head(10))

print(f"\nLow-variance output:\n{LOW_VARIANCE_FILE}")
print(f"\nF-score ranking output:\n{FSCORE_FILE}")
print(f"\nTop feature dataset output:\n{TOP_FEATURE_FILE}")


FileNotFoundError: 
Input file not found:
/content/outputs/02_recA_modeling_matrix.csv

Please run 02 fingerprint matrix assembly first.

## Original Uploaded Notebook Content

The cells below preserve the content from `03_Features_selection.ipynb`.

# 3. Feature Selection

This notebook performs low-variance filtering and ANOVA F-score feature ranking for the *Mycobacterium tuberculosis* RecA binary fingerprint dataset.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

INPUT_FILE = Path('outputs/02_recA_modeling_matrix.csv')
OUTPUT_DIR = Path('outputs/feature_selection')

LOW_VARIANCE_FILE = OUTPUT_DIR / '03_recA_low_variance_filtered.csv'
FSCORE_FILE = OUTPUT_DIR / '03_recA_fscore_ranking.csv'
TOP_FEATURE_FILE = OUTPUT_DIR / '03_recA_top_200_fscore.csv'

META_COLUMNS = ['Name', 'canonical_smiles', 'bioactivity_class', 'class']

variance_threshold = 0.16
top_k = 200

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 3.1 Load Dataset

In [ ]:
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Input file not found: {INPUT_FILE}')

df = pd.read_csv(INPUT_FILE)
df.shape

In [ ]:
df.head()

## 3.2 Prepare X and y

In [ ]:
missing = [col for col in META_COLUMNS if col not in df.columns]
if missing:
    raise ValueError(f'Missing metadata columns: {missing}')

meta = df[META_COLUMNS].copy()
feature_columns = [col for col in df.columns if col not in META_COLUMNS]

X = (
    df[feature_columns]
    .apply(pd.to_numeric, errors='coerce')
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = meta['class'].astype(int)

print('Original feature count:', X.shape[1])
print('Class distribution:')
print(meta['bioactivity_class'].value_counts())

## 3.3 Remove Low-Variance Features

In [ ]:
variance_selector = VarianceThreshold(threshold=variance_threshold)
X_low_variance_array = variance_selector.fit_transform(X)

selected_columns = X.columns[variance_selector.get_support()].tolist()

X_low_variance = pd.DataFrame(
    X_low_variance_array,
    columns=selected_columns,
    index=X.index
)

print('Variance threshold:', variance_threshold)
print('Features after low-variance filtering:', X_low_variance.shape[1])

## 3.4 Rank Features Using ANOVA F-score

In [ ]:
selector = SelectKBest(score_func=f_classif, k='all')
selector.fit(X_low_variance, y)

feature_scores = pd.DataFrame({
    'feature': X_low_variance.columns,
    'f_score': selector.scores_,
    'p_value': selector.pvalues_,
})

feature_scores = feature_scores.replace([np.inf, -np.inf], np.nan)
feature_scores['f_score'] = feature_scores['f_score'].fillna(-1.0)
feature_scores['p_value'] = feature_scores['p_value'].fillna(1.0)

feature_scores = feature_scores.sort_values(
    ['f_score', 'p_value'],
    ascending=[False, True]
).reset_index(drop=True)

feature_scores.head(20)

## 3.5 Build Top Feature Dataset

In [ ]:
top_k_final = min(top_k, len(feature_scores))
selected_features = feature_scores.head(top_k_final)['feature'].tolist()

top_feature_dataset = pd.concat(
    [meta, X_low_variance[selected_features]],
    axis=1
)

low_variance_dataset = pd.concat([meta, X_low_variance], axis=1)

low_variance_dataset.to_csv(LOW_VARIANCE_FILE, index=False)
feature_scores.to_csv(FSCORE_FILE, index=False)
top_feature_dataset.to_csv(TOP_FEATURE_FILE, index=False)

print('Top-k:', len(selected_features))
print('Saved low-variance dataset to:', LOW_VARIANCE_FILE)
print('Saved F-score ranking to:', FSCORE_FILE)
print('Saved top feature dataset to:', TOP_FEATURE_FILE)

## 3.6 Preview Final Selected Dataset

In [ ]:
top_feature_dataset.head()

## Stage 03a — RFE Feature-Selection Benchmark

The following cells contain the full converted Python workflow for SVM-RFE, RF-RFE, and XGBoost-RFE benchmarking.

In [ ]:
"""RFE benchmark stage for the RecA-TB feature-selection workflow.

Recent reference-backed methodology (2021-2025):
- Niazi and Mariam (2023), machine-learning-based chemoinformatics review.
- Qi et al. (2024), machine learning in drug discovery.
- Wang et al. (2024), comparative feature-selection strategies.
- Barrera-Garcia et al. (2024), feature selection systematic review.
- Rosenblatt et al. (2024), data leakage inflates prediction performance.
- Catacutan et al. (2024), machine learning in preclinical drug discovery.

This script benchmarks SVM-RFE, RF-RFE, and XGBoost-RFE on the cleaned
low-variance matrix, then evaluates downstream classifiers on a fixed holdout.

See feature_selection_recent_references.md for full citations and links.
"""

from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC
from xgboost import XGBClassifier


SCRIPT_DIR = Path(__file__).resolve().parent

INPUT_FILE = SCRIPT_DIR / "outputs" / "feature_selection" / "03_recA_low_variance_filtered.csv"
OUTPUT_DIR = SCRIPT_DIR / "outputs" / "rfe_benchmark"

META_COLUMNS = ["Name", "canonical_smiles", "bioactivity_class", "class"]

TEST_SIZE = 0.2
RANDOM_STATE = 42
N_FEATURES_TO_SELECT = 100
RFE_STEP = 0.2


def load_dataset() -> tuple[pd.DataFrame, pd.Series]:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(
            f"\nInput file not found:\n{INPUT_FILE}\n\n"
            "Please run 03_Feature_selection_low_variance_Fscore.py first."
        )

    df = pd.read_csv(INPUT_FILE)

    missing = [col for col in META_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"Missing metadata columns: {missing}")

    x = (
        df.drop(columns=META_COLUMNS)
        .apply(pd.to_numeric, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    y = df["class"].astype(int)

    return x, y


def get_selector(method_name: str) -> RFE:
    if method_name == "SVM-RFE":
        estimator = LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            dual="auto",
            max_iter=5000,
        )

    elif method_name == "RF-RFE":
        estimator = RandomForestClassifier(
            n_estimators=150,
            random_state=RANDOM_STATE,
            class_weight="balanced_subsample",
            n_jobs=-1,
        )

    elif method_name == "XGBoost-RFE":
        estimator = XGBClassifier(
            n_estimators=150,
            max_depth=5,
            learning_rate=0.08,
            subsample=0.8,
            colsample_bytree=0.9,
            min_child_weight=3,
            reg_lambda=2,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )

    else:
        raise ValueError(f"Unknown selector method: {method_name}")

    return RFE(
        estimator=estimator,
        n_features_to_select=N_FEATURES_TO_SELECT,
        step=RFE_STEP,
    )


def get_models() -> dict[str, object]:
    return {
        "SVM": Pipeline(
            [
                ("scaler", StandardScaler()),
                (
                    "model",
                    SVC(
                        kernel="rbf",
                        C=2,
                        gamma="scale",
                        probability=True,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "RF": RandomForestClassifier(
            n_estimators=300,
            min_samples_split=5,
            min_samples_leaf=4,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.7,
            colsample_bytree=1.0,
            min_child_weight=3,
            reg_lambda=2,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
    }


def predict_scores(model, x: pd.DataFrame) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]

    if hasattr(model, "decision_function"):
        scores = np.asarray(model.decision_function(x), dtype=float)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)

    raise ValueError("Model does not support probability or decision scoring.")


def metric_row(
    y_true: pd.Series,
    y_pred: np.ndarray,
    y_score: np.ndarray,
) -> dict[str, float]:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
    }


def plot_confusion_matrix(
    cm: np.ndarray,
    title: str,
    output_file: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(6.8, 5.5))

    image = ax.imshow(cm, interpolation="nearest")
    plt.colorbar(image, ax=ax)

    row_sums = cm.sum(axis=1, keepdims=True)

    percentages = np.divide(
        cm,
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j,
                i,
                f"{cm[i, j]}\n{percentages[i, j] * 100:.2f}%",
                ha="center",
                va="center",
                fontsize=11,
            )

    ax.set(
        xticks=[0, 1],
        yticks=[0, 1],
        xticklabels=["Inactive", "Active"],
        yticklabels=["Inactive", "Active"],
        ylabel="Actual Label",
        xlabel="Predicted Label",
        title=title,
    )

    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_roc(
    y_true: pd.Series,
    y_score: np.ndarray,
    title: str,
    output_file: Path,
) -> None:
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = roc_auc_score(y_true, y_score)

    fig, ax = plt.subplots(figsize=(6.8, 5.5))

    ax.plot(fpr, tpr, lw=2, label=f"ROC curve, AUC = {roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], lw=2, linestyle="--")

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.legend(loc="lower right")

    plt.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(fig)


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    x, y = load_dataset()

    if x.shape[1] < N_FEATURES_TO_SELECT:
        raise ValueError(
            f"Number of available features ({x.shape[1]}) is smaller than "
            f"N_FEATURES_TO_SELECT ({N_FEATURES_TO_SELECT})."
        )

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    train_rows: list[dict] = []
    test_rows: list[dict] = []
    selected_feature_map: dict[str, list[str]] = {}
    best_result: dict | None = None

    for method_name in ["SVM-RFE", "RF-RFE", "XGBoost-RFE"]:
        print(f"\nRunning {method_name}...")

        selector = get_selector(method_name)
        selector.fit(x_train, y_train)

        selected_columns = x_train.columns[selector.support_].tolist()
        selected_feature_map[method_name] = selected_columns

        x_train_sel = x_train[selected_columns].copy()
        x_test_sel = x_test[selected_columns].copy()

        for model_name, model in get_models().items():
            print(f"  Training {model_name}...")

            model.fit(x_train_sel, y_train)

            y_train_pred = model.predict(x_train_sel)
            y_train_score = predict_scores(model, x_train_sel)

            train_metrics = metric_row(
                y_train,
                y_train_pred,
                y_train_score,
            )

            train_rows.append(
                {
                    "feature_selection_method": method_name,
                    "model_algorithm": model_name,
                    **train_metrics,
                }
            )

            y_test_pred = model.predict(x_test_sel)
            y_test_score = predict_scores(model, x_test_sel)

            test_metrics = metric_row(
                y_test,
                y_test_pred,
                y_test_score,
            )

            result_row = {
                "feature_selection_method": method_name,
                "model_algorithm": model_name,
                **test_metrics,
            }

            test_rows.append(result_row)

            if best_result is None or result_row["roc_auc"] > best_result["roc_auc"]:
                best_result = {
                    **result_row,
                    "selected_columns": selected_columns,
                    "y_test_pred": y_test_pred,
                    "y_test_score": y_test_score,
                }

    train_df = pd.DataFrame(train_rows)
    test_df = pd.DataFrame(test_rows)

    train_df.to_csv(
        OUTPUT_DIR / "03a_rfe_train_metrics.csv",
        index=False,
    )

    test_df.to_csv(
        OUTPUT_DIR / "03a_rfe_test_metrics.csv",
        index=False,
    )

    with open(
        OUTPUT_DIR / "03a_selected_features_by_method.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(selected_feature_map, file, indent=2)

    if best_result is not None:
        cm = confusion_matrix(
            y_test,
            best_result["y_test_pred"],
        )

        title_prefix = (
            f"{best_result['model_algorithm']} "
            f"with {best_result['feature_selection_method']}"
        )

        plot_confusion_matrix(
            cm,
            f"Confusion Matrix: {title_prefix}",
            OUTPUT_DIR / "03a_best_rfe_confusion_matrix.png",
        )

        plot_roc(
            y_test,
            best_result["y_test_score"],
            f"ROC Curve: {title_prefix}",
            OUTPUT_DIR / "03a_best_rfe_roc_curve.png",
        )

        best_summary = {
            key: value
            for key, value in best_result.items()
            if key not in {
                "selected_columns",
                "y_test_pred",
                "y_test_score",
            }
        }

        pd.DataFrame([best_summary]).to_csv(
            OUTPUT_DIR / "03a_best_rfe_result.csv",
            index=False,
        )

    print("\nTrain metrics:")
    print(train_df.to_string(index=False))

    print("\nTest metrics:")
    print(test_df.to_string(index=False))

    print(f"\nOutputs saved to:\n{OUTPUT_DIR}")



In [ ]:

# Notebook runner for Stage 03a: RFE benchmark
# This may take time depending on number of descriptors and hardware.
main()
